In [1]:
import numpy as np
from ldpc.bposd_decoder import BpOsdDecoder
from ldpc.bplsd_decoder import BpLsdDecoder
from ldpc.bp_decoder import BpDecoder
from tqdm import tqdm
from scipy.sparse import csc_matrix
import stim
from typing import List, FrozenSet, Dict
import pandas as pd
import warnings
# from decoder import SSFDecoder
import matplotlib.pyplot as plt
import itertools


In [2]:
import sys 
import os

# Get absolute path to ../src (the folder containing qldpc_circuit)
src_path = os.path.abspath(os.path.join(os.path.dirname('/Users/ariannameinking/Documents/Brown_Research/quits/docs/ssf_circuit_level.ipynb'), '..', 'src'))

if src_path not in sys.path:
    sys.path.insert(0, src_path)


from quits.qldpc_code import *
from quits.circuit import get_qldpc_mem_circuit
from quits.simulation import get_stim_mem_result
from quits.decoder import SSFDecoder, sliding_window_circuit_mem
from quits.ldpc_utility import *

In [3]:
n1 = 625
n2 = 900

nodes = 12    # Number of variable nodes
dv = 3        # Variable node degree
dc = 4        # Check node degree (m = n * dv / dc should be an integer)
dist = 6      # Code distance


h1 = np.loadtxt('/Users/ariannameinking/Documents/Brown_Research/qldpc-circuit/parity_check_matrices/hgp%d.txt'%n1, dtype=int)
h2 = np.loadtxt('/Users/ariannameinking/Documents/Brown_Research/qldpc-circuit/parity_check_matrices/hgp%d.txt'%n2, dtype=int)
h3 = np.loadtxt('/Users/ariannameinking/Documents/Brown_Research/qldpc-circuit/parity_check_matrices/n=%d_dv=%d_dc=%d_dist=%d.txt'%(nodes, dv, dc, dist), dtype=int)

In [4]:
p_list = np.logspace(-2, -1, 4)


num_trials_list = np.array([500, 250, 100, 50],dtype=int)
num_rounds = 18
basis = 'Z'
W = 1
F = 1
h_L = [h1, h2, h3]
# 625 took 74 min, 0.778 it/sec 
# 900 took 190 min,  0.39 it/sec
# 225 took  20min, 2.91 it/sec

code_L = []
seed_L = [1,1,22]

for i,h in enumerate(h_L):
    code = HgpCode(h,h)
    code.build_graph(seed=seed_L[i])
    code_L.append(code)



df_BP = pd.DataFrame({'p': p_list, 'pL_625':np.zeros(len(p_list)),'pL_900':np.zeros(len(p_list)), 'pL_225':np.zeros(len(p_list))})

In [ ]:
for j, code in enumerate(code_L):
    for i, p in enumerate(p_list):
        print(f"code = {df_BP.columns[j+1][3:]}, p = {p}")
        p = p_list[i]
        num_trials = num_trials_list[i]

        circuit = stim.Circuit(get_qldpc_mem_circuit(code, p, p, p, p, num_rounds))
        zcheck_samples, logical_obs_samples = get_stim_mem_result(circuit, num_trials, seed=0)

        dict1 = {'code':code,'p':p, 'error_type':basis}
        dict2 = {'code':code,'p':p, 'error_type':basis}
        
        logical_pred = sliding_window_circuit_mem(zcheck_samples=zcheck_samples, circuit=circuit, hz=code.hz, lz=code.lz, W=W, F=F, decoder1=SSFDecoder, decoder2=SSFDecoder,dict1=dict1, dict2=dict2, error_rate_name1='p', error_rate_name2='p', function_name1='decode', function_name2='decode', tqdm_on=True)

        pL = np.sum((logical_obs_samples - logical_pred).any(axis=1)) / num_trials
        print('p: %.7f, pL: %.7f'%(p, pL))
            
        df_BP.loc[i, df_BP.columns[j+1]] = pL

code = 625, p = 0.01
